# Notebook 04: Classical ML Baselines (TF-IDF)
## SAMATRIX ResumeForge 2026 — Multiclass Resume Classification

### Objective
Systematic evaluation of classical linear classifiers on TF-IDF representations:
1. **Multinomial Naive Bayes** (fast probabilistic baseline)
2. **Logistic Regression** (with and without balanced class weights)
3. **Linear Support Vector Classifier (LinearSVC)** (margin maximization)
4. N-gram range tuning: `(1,1)`, `(1,2)`, and `(1,3)`
5. Hyperparameter tuning over regularizer `C`.


In [1]:
import os
import sys
import pandas as pd
import numpy as np
import joblib

# Robust project root setup (works from notebooks/ or workspace root)
if os.path.exists('src'):
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists('../src'):
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import REPORTS_DIR, MODELS_DIR, VECTORIZERS_DIR

# Load model comparison
comp_path = os.path.join(REPORTS_DIR, 'classical_model_comparison.csv')
if os.path.exists(comp_path):
    cml_df = pd.read_csv(comp_path)
    print("Classical Model Comparison Table:")
    display(cml_df)
else:
    print("Run python -m src.pipeline first.")


Classical Model Comparison Table:


,model,features,accuracy,macro_f1,weighted_f1,training_time
0,MultinomialNB,tfidf_uni_bi,0.524194,0.435350,0.476151,0.025579
1,LogisticRegression,tfidf_uni_bi,0.642473,0.587377,0.623245,1.319813
2,LogisticRegression_nobalance,tfidf_uni_bi,0.620968,0.545468,0.593959,1.423760
3,LinearSVC,tfidf_uni_bi,0.690860,0.625183,0.668870,0.609666
4,LinearSVC_nobalance,tfidf_uni_bi,0.680108,0.615954,0.659360,0.608110
5,LinearSVC,tfidf_uni,0.696237,0.634918,0.677364,0.347506
6,LinearSVC,tfidf_uni_bi_tri,0.677419,0.614199,0.658517,0.687605


### Model Interpretability: Top Discriminative Terms per Class
Linear classifiers offer direct interpretability through feature weight inspection (`model.coef_`).


In [2]:
interp_path = os.path.join(REPORTS_DIR, 'model_interpretability.json')
if os.path.exists(interp_path):
    import json
    with open(interp_path) as f:
        interp = json.load(f)
    for cat in ['INFORMATION-TECHNOLOGY', 'HEALTHCARE', 'CHEF', 'AVIATION', 'FINANCE']:
        if cat in interp:
            terms = [f"{t} ({s})" for t, s in interp[cat][:6]]
            print(f"{cat}: {', '.join(terms)}")


INFORMATION-TECHNOLOGY: technology (2.2111), information (1.3009), it (1.0752), security (1.0744), network (0.9289), infrastructure (0.9285)
HEALTHCARE: healthcare (4.0528), care (0.8744), medical (0.8162), services (0.7337), clinic (0.7244), medicine (0.6773)
CHEF: chef (3.2015), culinary (1.2799), kitchen (1.2524), cooking (0.8965), food (0.8374), recipes (0.7486)
AVIATION: aviation (3.4122), aircraft (2.6437), flight (1.3117), navy (0.8553), maintenance (0.6604), ammunition (0.6313)
FINANCE: finance (3.4334), financial (0.9948), accounting (0.8509), dealership (0.7228), forecasting (0.72), director (0.6209)
